## Missing Value Imputation Menggunakan WKNN


Imputasi missing value merupakan proses untuk mengisi data yang hilang agar dataset tetap dapat digunakan dalam proses analisis. Salah satu metode yang sering digunakan adalah Weighted K-Nearest Neighbor (WKNN).

Metode WKNN bekerja dengan mencari beberapa data yang memiliki kemiripan paling dekat dengan data yang memiliki nilai kosong. Kedekatan tersebut dihitung menggunakan jarak tertentu, seperti Euclidean Distance. Selanjutnya, nilai yang hilang akan diperkirakan berdasarkan nilai dari data tetangga tersebut.

Berbeda dengan metode KNN biasa, WKNN memberikan bobot pada setiap tetangga berdasarkan jaraknya. Data yang lebih dekat akan memiliki pengaruh lebih besar dibandingkan data yang lebih jauh, sehingga hasil prediksi menjadi lebih akurat.

Dengan menggunakan metode ini, missing value dapat diisi secara lebih optimal karena mempertimbangkan hubungan antar data.

**1. Data Awal**

Dataset yang digunakan memiliki tiga atribut yaitu IPK, PO, dan JML. Pada baris ke-7 terdapat nilai JML yang kosong, sehingga perlu dilakukan prediksi menggunakan metode Weighted K-Nearest Neighbor (WKNN).

| No  | IPK | PO     | JML |
|---- |-----|--------|-----|
| 1   | 2   | 200000 | 2   |
| 2   | 3   | 300000 | 3   |
| 3   | 4   | 200000 | 2   |
| 4   | 2   | 200000 | 3   |
| 5   | 3   | 300000 | 2   |
| 6   | 4   | 400000 | 3   |
| 7   | 2   | 300000 | ?   |


Nilai JML pada data ke-7 belum diketahui, sehingga akan diperkirakan berdasarkan kemiripan dengan data lain.

**2. Normalisasi Data (Min-Max Scaling)**

Sebelum menghitung jarak, data perlu diseragamkan skalanya menggunakan normalisasi Min-Max agar tidak bias.

Rumus:

$$
X' = \frac{X - X_{\min}}{X_{\max} - X_{\min}}
$$

Nilai Minimum & Maksimum

| Atribut | Min | Max |
|--------|-----|-----|
| IPK    | 2   | 4   |
| PO     | 200000 | 400000 |
| JML    | 2   | 3   |


Hasil Normalisasi

| IPK | PO  | JML |
|-----|-----|-----|
| 0   | 0   | 0   |
| 0.5 | 0.5 | 1   |
| 1   | 0   | 0   |
| 0   | 0   | 1   |
| 0.5 | 0.5 | 0   |
| 1   | 1   | 1   |
| 0   | 0.5 | ?   |

Data yang ingin diprediksi:

IPK = 0

PO = 0.5

**3. Menghitung Jarak (Euclidean Distance)**

Jarak dihitung menggunakan rumus:

$$
d = \sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}
$$

Perhitungan jarak dari data ke-7 ke data lainnya:

| Data | Jarak | JML |
|------|------|-----|
| 1    | 0.5  | 2   |
| 2    | 0.5  | 3   |
| 3    | 1.118| 2   |
| 4    | 0.5  | 3   |
| 5    | 0.5  | 2   |
| 6    | 1.118| 3   |


**4. Menentukan Nilai K**

Misalkan digunakan K = 3, maka diambil 3 data dengan jarak paling kecil.

Data terdekat:

- Data 1 → JML = 2

- Data 2 → JML = 3

- Data 4 → JML = 3

**5. Perhitungan WKNN**

Pada metode WKNN, setiap tetangga memiliki bobot yang dihitung dari jaraknya:


$$
w = \frac{1}{d}
$$


Karena semua jarak = 0.5, maka bobotnya sama:


$$
w = \frac{1}{0.5} = 2
$$


Tabel Perhitungan:

| Data | Bobot | JML |
|------|------|-----|
| 1    | 2    | 2   |
| 2    | 2    | 3   |
| 4    | 2    | 3   |


Prediksi:


$$
JML = \frac{(2 \times 2) + (2 \times 3) + (2 \times 3)}{2 + 2 + 2}
$$

$$
JML = \frac{4 + 6 + 6}{6}
$$

$$
JML = 2.67
$$

$$
JML = 3
$$

Dibulatkan menjadi:

JML = 3

**6. Data Setelah Imputasi**

| No | IPK | PO     | JML |
|----|-----|--------|-----|
| 1  | 2   | 200000 | 2   |
| 2  | 3   | 300000 | 3   |
| 3  | 4   | 200000 | 2   |
| 4  | 2   | 200000 | 3   |
| 5  | 3   | 300000 | 2   |
| 6  | 4   | 400000 | 3   |
| 7  | 2   | 300000 | 3   |


Nilai yang sebelumnya kosong berhasil diperkirakan menggunakan metode WKNN.

**7. Implementasi Menggunakan Python**

Berikut contoh implementasi menggunakan scikit-learn:







In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.neighbors import KNeighborsRegressor

# dataset
data = {
    'IPK':[2,3,4,2,3,4,2],
    'PO':[200000,300000,200000,200000,300000,400000,300000],
    'JML':[2,3,2,3,2,3,np.nan]
}

df = pd.DataFrame(data)

# normalisasi
scaler = MinMaxScaler()
df[['IPK','PO']] = scaler.fit_transform(df[['IPK','PO']])

# pisahkan data
train = df[df['JML'].notna()]
test = df[df['JML'].isna()]

X_train = train[['IPK','PO']]
y_train = train['JML']

model = KNeighborsRegressor(n_neighbors=3, weights='distance')
model.fit(X_train, y_train)

prediksi = model.predict(test[['IPK','PO']])

print("Hasil prediksi JML:", prediksi[0])

Hasil prediksi JML: 2.3333333333333335


**8. Analisis Perbedaan Hasil**

Hasil manual memberikan nilai 3, sedangkan hasil dari program bisa menghasilkan nilai yang sedikit berbeda.

Hal ini terjadi karena:

- Terdapat beberapa data dengan jarak yang sama

- Program memilih tetangga berdasarkan urutan data

- Perhitungan bobot dilakukan secara otomatis dan lebih presisi

Sehingga nilai akhir dari program bisa tidak persis sama dengan perhitungan manual, meskipun tetap mendekati.